In [138]:
# Install libraries
!pip install rasterio
!pip install geopandas

In [139]:
# Load the data from the repository
!curl -L https://github.com/a-taylor1/DATA450_Team5/raw/f96e6ea78ed178af4fb4406b0d911c6bb98e1e0b/test_data.zip --output test_data.zip
!unzip -qo test_data.zip
%cd LF2024_EVC_CONUS/
# Print all of the files in the folder
!ls -l

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 2814k  100 2814k    0     0  2170k      0  0:00:01  0:00:01 --:--:-- 5371k
/content/LF2024_EVC_CONUS/LF2024_EVC_CONUS/LF2024_EVC_CONUS/LF2024_EVC_CONUS/LF2024_EVC_CONUS
total 9560
-rwxrwxrwx 1 root root  103426 Apr  7 00:53 LF2024_EVC_CONUS.dbf
-rwxrwxrwx 1 root root     337 Apr  7 00:53 LF2024_EVC_CONUS.GeoJSON
-rwxrwxrwx 1 root root      93 Apr  7 00:53 LF2024_EVC_CONUS.tfw
-rwxrwxrwx 1 root root 6726706 Apr  7 00:53 LF2024_EVC_CONUS.tif
-rwxrwxrwx 1 root root 2819920 Apr  7 00:53 LF2024_EVC_CONUS.tif.ovr
-rwxrwxrwx 1 root root   62224 Apr  7 00:53 LF2024_EVC_CONUS.tif.vat.dbf
-rwxrwxrwx 1 root root    1622 Apr  7 00:53 metadata.dbf
-rwxrwxrwx 1 root root     459 Apr  7 00:53 metadata.prj
-rwxrwxrwx 1 root root    2132 Apr  7 00:53 metadata.shp
-r

In [140]:
import rasterio
import pandas as pd

# Open the LANDFIRE raster file
with rasterio.open('LF2024_EVC_CONUS.tif') as dataset:
  for i in dataset.indexes:
    band = dataset.read(i)
    print(f"Band {i}: {band}")

Band 1: [[32767 32767 32767 ... 32767 32767 32767]
 [32767 32767 32767 ... 32767 32767 32767]
 [32767 32767 32767 ... 32767 32767 32767]
 ...
 [32767 32767 32767 ... 32767 32767 32767]
 [32767 32767 32767 ... 32767 32767 32767]
 [32767 32767 32767 ... 32767 32767 32767]]


In [141]:
# IMPORT DATAFRAME OF EV CODES
from geopandas import read_file

key_table:pd.DataFrame = read_file('LF2024_EVC_CONUS.dbf')
key_table = key_table.drop(columns = ['R', 'G', 'B', 'RED', 'GREEN', 'BLUE'])
key_table

,VALUE,CLASSNAMES
0,-9999,Fill-NoData
1,11,Open Water
2,12,Snow/Ice
3,13,Developed-Upland Deciduous Forest
4,14,Developed-Upland Evergreen Forest
...,...,...
288,395,Herb Cover = 95%
289,396,Herb Cover = 96%
290,397,Herb Cover = 97%
291,398,Herb Cover = 98%


### **SORT CLASSNMES INTO UNIQUE COLUMNS**

In [142]:
# 1. Get all levels of CLASSNAMES
class_levels = key_table['CLASSNAMES'].to_list()
class_levels[1:10]

['Open Water',
 'Snow/Ice',
 'Developed-Upland Deciduous Forest',
 'Developed-Upland Evergreen Forest',
 'Developed-Upland Mixed Forest',
 'Developed-Upland Herbaceous',
 'Developed-Upland Shrubland',
 'Developed - Open Space',
 'Developed-Low Intensity']

In [143]:
# 2. Separate continuous and discrete classes
#   NOTE: The dataframe represents continuous classes in the form "class name = xx%"
continuous_classes = []
discrete_classes = []
for c in class_levels:
  try:
    c.index('=')
    continuous_classes.append(c)
  except:
    discrete_classes.append(c)

print(f'CONTINUOUS:\n{continuous_classes[1:10]}')
print(f'CONTINUOUS:\n{discrete_classes[1:10]}')

CONTINUOUS:
['Tree Cover = 11%', 'Tree Cover = 12%', 'Tree Cover = 13%', 'Tree Cover = 14%', 'Tree Cover = 15%', 'Tree Cover = 16%', 'Tree Cover = 17%', 'Tree Cover = 18%', 'Tree Cover = 19%']
CONTINUOUS:
['Open Water', 'Snow/Ice', 'Developed-Upland Deciduous Forest', 'Developed-Upland Evergreen Forest', 'Developed-Upland Mixed Forest', 'Developed-Upland Herbaceous', 'Developed-Upland Shrubland', 'Developed - Open Space', 'Developed-Low Intensity']


In [155]:
# 3. Create columns for continuous classes

# a) Map the name of each class to an ordered list of values in the dataset

def translate_classname(name:str) -> tuple[str, str]:
  split_c = [name]
  if name in continuous_classes:
    if ' = ' in name: split_c = name.split(' = ')
    elif ' >= ' in name: split_c = name.split(' >= ')
    split_c[0] = split_c[0].replace(' ', '_')
    split_c[1] = int(split_c[1].replace('%', ''))
  else:
    split_c[0] = name.replace(' ', '_')
    split_c.append(True)
  return tuple(split_c)
def extract_classname(name:str) -> str:
  return translate_classname(name)[0].lower()
def extract_value(name:str) -> int|bool:
  return translate_classname(name)[1]

name_series = key_table['CLASSNAMES'].apply(extract_classname).rename('class_name')
value_series = key_table['CLASSNAMES'].apply(extract_value).rename('class_value')
result_df = pd.concat([name_series, value_series], axis=1)
result_df
# b) Iterate across the class names, populate a list for each
# c) Add each series to the dataframe

# 4. Create dummies for discrete classes


,class_name,class_value
0,fill-nodata,True
1,open_water,True
2,snow/ice,True
3,developed-upland_deciduous_forest,True
4,developed-upland_evergreen_forest,True
...,...,...
288,herb_cover,95
289,herb_cover,96
290,herb_cover,97
291,herb_cover,98


In [ ]:
!pip install raster2xyz
